In [20]:
from src.toolsjasinski._parse_video import parse_video
from src.toolsjasinski._split_ALEX import split_ALEX
import numpy as np
import matplotlib.pyplot as plt
import csv

# File Loading

In [ ]:
path = r'src\toolsjasinski\tests\ALEX_Nanowells_10mWRL_50mWGL_3MM_2min.nd2'

video = parse_video(path)

print(video)
print(f"File Size: {video.nbytes/1e9:.2f} Gb")
print(video.sizes)
print('pixel-size', video['X'].values[1], video['X'].attrs['units'])

# ALEX Splitting and xr. Dataset

In [ ]:
alex_ds = split_ALEX(video)

# Tracking

In [ ]:
import trackpy as tp
#tracking
trackDD = tp.batch(alex_ds["donorCam_donor"].values, diameter=5)
trackAD = tp.batch(alex_ds["acceptorCam_donor"].values, diameter=5)
# linking
t = tp.link(trackDD, 5, memory = 3)
f = tp.link(trackAD, 5, memory = 3)
# filter
t1 = tp.filter_stubs(t, 100)
f1 = tp.filter_stubs(f, 100)
# plotting
t1.head()
tp.plot_traj(t1)

Frame 792: 2657 features


# Saving as an Output

In [ ]:
class ParticleClass():
    def __init__(self, video_index, particle_index, xy_red, xy_green, intensity_red, intensity_green) -> None:
        self.video_index     = video_index
        self.particle_index  = particle_index
        self.xy_red          = xy_red
        self.xy_green        = xy_green
        self.intensity_red   = intensity_red
        self.intensity_green = intensity_green

class Output():
    def __init__(self) -> None:
        self.date = None
        self.particle_array = []
    
    def add_particle(self, particle) -> None:
        assert isinstance(particle, ParticleClass)
        self.particle_array.append(particle)

    def serialize_to_csv(self, filepath: str) -> None:
        headers = ['date', 'video_index', 'particle_index', 'xy_red', 'xy_green', 'intensity_red', 'intensity_green']

        with open(filepath, mode='w', newline='', encoding = 'utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(headers)

            for particle in self.particle_array:
                writer.writerow([
                    self.date,
                    particle.video_index,
                    particle.particle_index,
                    particle.xy_red,
                    particle.xy_green,
                    particle.intensity_red,
                    particle.intensity_green
                ])

In [ ]:
out = Output()

# Loop using iterrows
for index, row in t1.iterrows():
    particle = ParticleClass(1, index, (row['x'], row['y']), (None), row['mass'], None)
    out.add_particle(particle)

In [ ]:
import csv
out.serialize_to_csv('test_csv.csv')